# Laboratório — KNN, escalas e alta dimensionalidade

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/03-machine-learning/notebooks/07-knn-distancias-dimensionalidade-laboratorio.ipynb)

Neste laboratório você vai verificar, e não apenas aceitar, quatro afirmações:

1. KNN reproduzível pode ser implementado com cálculo de distâncias, ordenação e voto;
2. unidades incompatíveis mudam a vizinhança;
3. `k` e pesos devem ser escolhidos apenas nos dados de desenvolvimento;
4. dimensões irrelevantes diluem a noção de proximidade.

O teste fica isolado até a escolha final. As figuras têm descrição textual imediatamente abaixo ou antes da célula que as produz.

## Ambiente e reprodutibilidade

Dependências mínimas: Python 3.10, NumPy 1.24, pandas 1.5, Matplotlib 3.7 e scikit-learn 1.3.

Os dados são sintéticos e documentados. A semente `20260908` controla geração, divisão e validação cruzada. Não há downloads, credenciais nem estado externo.

In [ ]:
import platform
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import make_moons
from sklearn.dummy import DummyClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260908
np.set_printoptions(precision=6, suppress=True)

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Matplotlib:", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)

## 1. Dados, unidade de análise e protocolo

Cada linha representa uma observação independente com duas características e um rótulo binário. `make_moons` cria uma fronteira curva, adequada para um método local. Multiplicaremos a primeira característica por 1.000 para representar, por exemplo, uma coluna em milímetros e outra em metros.

Primeiro separamos 25% como teste final. Todo ajuste de `k`, pesos e análise de ablação ocorre nos 75% restantes (`X_dev`).

In [ ]:
X_base, y = make_moons(n_samples=1_000, noise=0.24, random_state=SEED)
indices = np.arange(len(y))
idx_dev, idx_test = train_test_split(
    indices, test_size=0.25, stratify=y, random_state=SEED
)

X_units = X_base.copy()
X_units[:, 0] *= 1_000.0
X_dev, X_test = X_units[idx_dev], X_units[idx_test]
y_dev, y_test = y[idx_dev], y[idx_test]

assert len(idx_dev) == 750 and len(idx_test) == 250
assert set(idx_dev).isdisjoint(idx_test)
assert np.bincount(y_dev).tolist() == [375, 375]
print("Desenvolvimento:", X_dev.shape, "| Teste reservado:", X_test.shape)
print("Amplitude aproximada por feature:", np.ptp(X_dev, axis=0))

**Figura 1 — descrição acessível:** dispersão dos 750 exemplos de desenvolvimento. As duas classes formam arcos intercalados; o eixo horizontal está em uma escala cerca de mil vezes maior que o vertical.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
scatter = ax.scatter(X_dev[:, 0], X_dev[:, 1], c=y_dev, cmap="coolwarm", s=18, alpha=0.75)
ax.set(xlabel="feature 0 (unidade × 1.000)", ylabel="feature 1", title="Dados de desenvolvimento: duas classes não lineares")
ax.grid(alpha=0.2)
plt.show()

## 2. KNN do zero em um exemplo pequeno

Para uma consulta `q`, calculamos todas as distâncias euclidianas, ordenamos os índices e votamos entre os `k` primeiros. Empates precisam de uma regra explícita. A função abaixo escolhe a menor classe em caso de empate, a mesma convenção de `argmax` sobre classes ordenadas.

In [ ]:
def knn_manual(X_train, y_train, query, k):
    distances = np.sqrt(np.sum((X_train - query) ** 2, axis=1))
    order = np.argsort(distances, kind="stable")[:k]
    classes, counts = np.unique(y_train[order], return_counts=True)
    prediction = classes[np.argmax(counts)]
    return prediction, order, distances[order], counts / k

X_small = np.array([[1., 2.], [3., 1.], [8., 8.], [2., 2.], [7., 7.]])
y_small = np.array([0, 1, 1, 0, 1])
q = np.array([1., 1.])
pred, neighbors, distances, proportions = knn_manual(X_small, y_small, q, k=3)

print("Índices vizinhos:", neighbors.tolist())
print("Distâncias:", distances)
print("Proporções por classe:", proportions)
print("Predição:", int(pred))
assert neighbors.tolist() == [0, 3, 1]
assert pred == 0
assert np.allclose(distances, [1.0, np.sqrt(2), 2.0])

## 3. Escala, `k` e pesos sem consultar o teste

Usaremos validação cruzada estratificada com cinco folds nos dados de desenvolvimento. O `StandardScaler` permanece dentro do `Pipeline`: em cada fold, média e desvio-padrão são aprendidos somente na parte de treino daquele fold.

Além de `k`, comparamos voto uniforme e voto inversamente proporcional à distância. A métrica de seleção é acurácia balanceada; ela dá o mesmo peso às classes mesmo que, em outro problema, haja desbalanceamento.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
k_values = [1, 3, 5, 9, 15, 25, 41]

scaled_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("knn", KNeighborsClassifier()),
])
grid = GridSearchCV(
    scaled_pipe,
    param_grid={"knn__n_neighbors": k_values, "knn__weights": ["uniform", "distance"]},
    scoring="balanced_accuracy",
    cv=cv,
    n_jobs=1,
    return_train_score=True,
)
grid.fit(X_dev, y_dev)

cv_results = (
    pd.DataFrame(grid.cv_results_)[
        ["param_knn__n_neighbors", "param_knn__weights", "mean_train_score", "mean_test_score", "std_test_score"]
    ]
    .rename(columns={
        "param_knn__n_neighbors": "k",
        "param_knn__weights": "pesos",
        "mean_train_score": "treino_medio",
        "mean_test_score": "validacao_media",
        "std_test_score": "validacao_dp",
    })
    .sort_values(["validacao_media", "k"], ascending=[False, True])
)
print(cv_results.head(8).to_string(index=False))
print("\nMelhor configuração:", grid.best_params_)
print("Acurácia balanceada CV:", f"{grid.best_score_:.6f}")

In [ ]:
unscaled_rows = []
for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k, weights="uniform")
    scores = cross_val_score(model, X_dev, y_dev, scoring="balanced_accuracy", cv=cv, n_jobs=1)
    unscaled_rows.append((k, scores.mean(), scores.std()))

unscaled = pd.DataFrame(unscaled_rows, columns=["k", "media", "dp"])
best_unscaled = unscaled.loc[unscaled["media"].idxmax()]
print(unscaled.to_string(index=False, float_format=lambda v: f"{v:.6f}"))
print("\nMelhor CV sem escala:", f"{best_unscaled['media']:.6f}")
print("Ganho do pipeline escalado:", f"{grid.best_score_ - best_unscaled['media']:.6f}")
assert grid.best_score_ > best_unscaled["media"] + 0.05

## 4. Avaliação final, uma única vez

Somente agora usamos o conjunto reservado. Comparamos o KNN escolhido com um `DummyClassifier` que sempre prevê a classe mais frequente. O teste não volta ao ciclo de decisão: seu papel é estimar desempenho, não selecionar hiperparâmetros.

In [ ]:
final_model = grid.best_estimator_
final_model.fit(X_dev, y_dev)
test_pred = final_model.predict(X_test)
test_score = balanced_accuracy_score(y_test, test_pred)

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_dev, y_dev)
dummy_score = balanced_accuracy_score(y_test, dummy.predict(X_test))

print("KNN — acurácia balanceada no teste:", f"{test_score:.6f}")
print("Baseline majoritária:", f"{dummy_score:.6f}")
print("Ganho absoluto:", f"{test_score - dummy_score:.6f}")
assert test_score > 0.85
assert np.isclose(dummy_score, 0.5)

## 5. Fronteiras locais e viés–variância

Para visualizar apenas a forma das fronteiras, voltamos às duas features originais e ajustamos modelos auxiliares. Estes gráficos não alteram a escolha nem a estimativa final.

**Figura 2 — descrição acessível:** três painéis mostram fronteiras de decisão no formato de luas. Com `k=1`, a borda tem pequenas ilhas e irregularidades; com `k` intermediário, acompanha os arcos; com `k=81`, fica excessivamente suave.

In [ ]:
def plot_boundary(ax, k, title):
    model = Pipeline([("scale", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=k))])
    model.fit(X_base[idx_dev], y_dev)
    x0 = np.linspace(X_base[:, 0].min() - 0.4, X_base[:, 0].max() + 0.4, 220)
    x1 = np.linspace(X_base[:, 1].min() - 0.4, X_base[:, 1].max() + 0.4, 180)
    xx, yy = np.meshgrid(x0, x1)
    zz = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.25, cmap="coolwarm")
    ax.scatter(X_base[idx_dev, 0], X_base[idx_dev, 1], c=y_dev, cmap="coolwarm", s=8, alpha=0.55)
    ax.set_title(title)
    ax.set_xlabel("feature 0")
    ax.set_ylabel("feature 1")

best_k = int(grid.best_params_["knn__n_neighbors"])
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for ax, k, title in zip(axes, [1, best_k, 81], ["k=1", f"k={best_k} (selecionado)", "k=81"]):
    plot_boundary(ax, k, title)
plt.show()

## 6. Dimensões irrelevantes

Agora mantemos o mesmo fenômeno e os mesmos folds, mas anexamos colunas gaussianas independentes do rótulo. A escala continua dentro do pipeline. Assim isolamos o efeito de adicionar dimensões sem sinal.

Usamos apenas validação cruzada em desenvolvimento — o teste reservado não será reutilizado.

In [ ]:
rng_noise = np.random.default_rng(SEED)
noise_bank = rng_noise.normal(size=(len(y), 200))
dimension_rows = []

for n_noise in [0, 10, 50, 200]:
    X_aug = np.c_[X_base, noise_bank[:, :n_noise]] if n_noise else X_base.copy()
    model = Pipeline([
        ("scale", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=best_k, weights=grid.best_params_["knn__weights"])),
    ])
    scores = cross_val_score(
        model, X_aug[idx_dev], y_dev, scoring="balanced_accuracy", cv=cv, n_jobs=1
    )
    dimension_rows.append((n_noise, X_aug.shape[1], scores.mean(), scores.std()))

dimension_results = pd.DataFrame(
    dimension_rows, columns=["features_ruido", "dimensoes_totais", "cv_media", "cv_dp"]
)
print(dimension_results.to_string(index=False, float_format=lambda v: f"{v:.6f}"))
assert dimension_results.iloc[-1]["cv_media"] < dimension_results.iloc[0]["cv_media"] - 0.10

**Figura 3 — descrição acessível:** linha descendente relaciona o número de features de ruído à acurácia balanceada média. A faixa vertical em cada ponto representa um desvio-padrão entre folds.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.errorbar(
    dimension_results["features_ruido"], dimension_results["cv_media"],
    yerr=dimension_results["cv_dp"], marker="o", capsize=4
)
ax.set(
    xlabel="features gaussianas irrelevantes",
    ylabel="acurácia balanceada (CV)",
    title="Features irrelevantes degradam a vizinhança",
)
ax.set_ylim(0.45, 1.0)
ax.grid(alpha=0.25)
plt.show()

## 7. Concentração das distâncias

Uma forma operacional de observar a maldição da dimensionalidade é o **contraste relativo**

\[
C = \frac{d_{\max}-d_{\min}}{d_{\min}}.
\]

Quanto menor `C`, menos o vizinho mais próximo se destaca do mais distante. A simulação abaixo mede o contraste mediano para 200 consultas, cada uma comparada a 1.000 pontos gaussianos.

In [ ]:
rng_dist = np.random.default_rng(SEED + 1)
contrast_rows = []
for d in [2, 10, 50, 200]:
    references = rng_dist.normal(size=(1_000, d))
    queries = rng_dist.normal(size=(200, d))
    contrasts = []
    nearest_over_mean = []
    for query in queries:
        ds = np.linalg.norm(references - query, axis=1)
        contrasts.append((ds.max() - ds.min()) / ds.min())
        nearest_over_mean.append(ds.min() / ds.mean())
    contrast_rows.append((d, np.median(contrasts), np.median(nearest_over_mean)))

contrast = pd.DataFrame(contrast_rows, columns=["dimensoes", "contraste_mediano", "dmin_sobre_media"])
print(contrast.to_string(index=False, float_format=lambda v: f"{v:.6f}"))
assert contrast.iloc[-1]["contraste_mediano"] < contrast.iloc[0]["contraste_mediano"]
assert contrast.iloc[-1]["dmin_sobre_media"] > contrast.iloc[0]["dmin_sobre_media"]

## 8. Auditando uma predição

`predict_proba` do KNN é o voto local normalizado (ou ponderado). Podemos recuperar os vizinhos para explicar a saída, lembrando que frequência local não garante probabilidade calibrada.

In [ ]:
query = X_test[[0]]
scaled_query = final_model.named_steps["scale"].transform(query)
distances, neighbor_ids = final_model.named_steps["knn"].kneighbors(scaled_query)
neighbor_labels = y_dev[neighbor_ids[0]]
proba = final_model.predict_proba(query)[0]

print("Distâncias padronizadas:", np.round(distances[0], 4).tolist())
print("Rótulos vizinhos:", neighbor_labels.tolist())
print("Probabilidades do modelo:", np.round(proba, 6).tolist())
print("Classe prevista:", int(final_model.predict(query)[0]))
assert np.isclose(proba.sum(), 1.0)
assert final_model.predict(query)[0] == final_model.classes_[np.argmax(proba)]

## 9. Conclusões verificadas

- O algoritmo manual encontrou os mesmos vizinhos esperados por cálculo geométrico.
- Escalar dentro do pipeline melhorou a seleção sem vazar estatísticas entre folds.
- `k` e pesos foram escolhidos apenas em desenvolvimento; o teste foi aberto uma vez.
- Features sem sinal reduziram a qualidade do método, mesmo padronizadas.
- Em alta dimensão, a distância mínima se aproximou da distância média e o contraste relativo caiu.

### Desafios

1. Troque a métrica por Manhattan (`p=1`) e compare-a por validação cruzada, sem tocar no teste.
2. Repita a ablação com ruído correlacionado. A degradação é igual à do ruído independente?
3. Em um problema desbalanceado, compare voto uniforme e por distância usando recall por classe.

As respostas conceituais e o protocolo correto estão na aula. Não reutilize o teste para escolher a alternativa vencedora.